# Лабораторная работа: семантическая сегментация поражений листа/плода клубники

Киберфизические системы в агроконтексте: автоматическое картирование симптомов по изображениям с беспилотных/стационарных камер позволяет локально оценивать распространённость болезни и планировать точечные обработки, снижая объём пестицидов.

**Среда:** ноутбук рассчитан на запуск в **Google Colab** (GPU рекомендуется). При локальном запуске установите зависимости из первой кодовой ячейки.

**Данные:** набор [Strawberry Disease Detection Dataset](https://www.kaggle.com/datasets/usmanafzaal/strawberry-disease-detection-dataset) загружается через `kagglehub` (см. раздел 1).

**Модели сегментации (п.2–3):** `segmentation_models.pytorch` (SMP); **`torchvision`** используется только для **`transforms`**.

## 1. Выбор начальных условий

### 1a) Набор данных и обоснование

Выбран датасет **Strawberry Disease Detection Dataset** (`usmanafzaal/strawberry-disease-detection-dataset` на Kaggle). Это **практическая агрономическая задача**: ранняя диагностика болезней клубники по визуальным признакам. Исходные аннотации ориентированы на **сегментацию поражений** (контуры/маски областей симптомов). Для выполнения формулировки ТЗ про **семантическую** сегментацию мы **объединяем все экземпляры одного класса болезни в один семантический класс на пиксельной маске** (не разделяем отдельные «острова» одной болезни как разные ID объектов). Фон и (при наличии) здоровая ткань выделяются отдельными классами.

**Уникальность:** фиксируется выбором конкретного публичного набора и сценария «агромониторинг + семантические маски симптомов».

### 1b) Метрики качества и обоснование

Для семантической сегментации используем:

1. **mIoU (mean Intersection-over-Union)** — стандарт для сегментации; устойчиво сравнивает перекрытие предсказанной и истинной областей класса, особенно при несбалансированных площадях классов.
2. **Mean Dice (F1 на уровне класса по маскам)** — хорошо коррелирует с «полезностью» маски при медицинской/биологической интерпретации (площадные поражения).
3. **Pixel Accuracy** — простая интерпретируемая величина, но может быть завышена при доминировании фона; поэтому **основной отчётной метрикой** считаем **mIoU**, а pixel accuracy — вспомогательной.

Ниже задаём единые функции метрик и общий цикл обучения/валидации.

In [ ]:
# Установка зависимостей (Colab и локально при необходимости)
%pip install -q kagglehub torch torchvision segmentation-models-pytorch torchmetrics tqdm pillow matplotlib

In [ ]:
import os
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import kagglehub

# Download latest version (требуются учётные данные Kaggle в Colab: см. комментарий ниже)
path = kagglehub.dataset_download("usmanafzaal/strawberry-disease-detection-dataset")
print("Path to dataset files:", path)

# Colab: при ошибке аутентификации Kaggle загрузите kaggle.json:
# from google.colab import files
# files.upload()  # выберите kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

### Загрузка данных и авто-обнаружение структуры

Наборы на Kaggle могут слегка отличаться по структуре папок. Ниже — **эвристический сканер**: ищет изображения и пары масок/аннотаций (PNG маски или COCO JSON). Если ваш архив отличается, при необходимости поправьте `IMAGE_EXTS` / корневую папку в одном месте.

In [ ]:
DATA_ROOT = Path(path)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def iter_images(root: Path):
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            yield p


def find_coco_jsons(root: Path) -> List[Path]:
    return [p for p in root.rglob("*.json") if "coco" in p.name.lower() or "annotations" in p.parts[-3:]]


def pick_image_roots(root: Path) -> List[Path]:
    preferred = ["images", "Images", "train", "Train", "JPEGImages"]
    dirs = []
    for name in preferred:
        d = root / name
        if d.is_dir():
            dirs.append(d)
    if dirs:
        return dirs
    # fallback: каталоги, где много картинок
    best = None
    best_cnt = 0
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        cnt = sum(1 for _ in iter_images(d))
        if cnt > best_cnt:
            best_cnt = cnt
            best = d
    return [best] if best is not None else [root]


image_roots = pick_image_roots(DATA_ROOT)
print("Candidate image roots:")
for r in image_roots:
    print(" -", r)

coco_jsons = find_coco_jsons(DATA_ROOT)
print("Found JSON candidates:", len(coco_jsons))
for j in coco_jsons[:5]:
    print(" -", j)

In [ ]:
def rasterize_coco_to_semantic(coco: dict, image_id: int, out_hw: Tuple[int, int], cat_ids_sorted: List[int]) -> np.ndarray:
    """Семантическая маска: для каждого пикселя — индекс в cat_ids_sorted или 0 (фон)."""
    h, w = out_hw
    mask = np.zeros((h, w), dtype=np.int64)
    cat_to_idx = {c: i + 1 for i, c in enumerate(cat_ids_sorted)}  # 0 — фон

    for ann in coco.get("annotations", []):
        if ann.get("image_id") != image_id:
            continue
        cat = ann.get("category_id")
        if cat not in cat_to_idx:
            continue
        seg = ann.get("segmentation")
        if not seg:
            continue
        m = Image.new("L", (w, h), 0)
        dr = ImageDraw.Draw(m)
        if isinstance(seg, list) and isinstance(seg[0], list):
            for poly in seg:
                if len(poly) >= 6:
                    dr.polygon(list(zip(poly[0::2], poly[1::2])), outline=1, fill=1)
        elif isinstance(seg, dict) and "counts" in seg:
            # RLE — пропускаем в упрощённом бейзлайне (можно добавить pycocotools)
            continue
        mm = np.array(m) > 0
        idx = cat_to_idx[cat]
        mask[mm] = idx
    return mask


class StrawberrySemanticDataset(Dataset):
    """Поддержка: (A) пары image/mask PNG; (B) COCO JSON + изображения."""

    def __init__(
        self,
        root: Path,
        image_size: int = 256,
        split: str = "train",
        max_samples: Optional[int] = None,
    ):
        self.root = root
        self.image_size = image_size
        self.items: List[Tuple[Path, Optional[Path], Optional[Path], Optional[int]]] = []
        # tuple: (image_path, mask_path_or_none, coco_json_or_none, coco_image_id)

        roots = pick_image_roots(root)
        # 1) Попытка: рядом лежат маски
        candidates = []
        for ir in roots:
            for img_path in sorted(iter_images(ir)):
                stem = img_path.stem
                mask_path = None
                for cand in [
                    img_path.with_name(stem + "_mask.png"),
                    img_path.with_name(stem + ".png"),
                    ir.parent / "masks" / (stem + ".png"),
                    ir.parent / "Masks" / (stem + ".png"),
                    ir.parent / "labels" / (stem + ".png"),
                ]:
                    if cand.is_file() and cand != img_path:
                        mask_path = cand
                        break
                candidates.append((img_path, mask_path, None, None))

        paired = [c for c in candidates if c[1] is not None]
        if len(paired) >= max(10, int(0.05 * len(candidates))):
            self.items = paired
            self.mode = "png"
        else:
            # 2) COCO
            coco_jsons_local = find_coco_jsons(root)
            coco_path = None
            if coco_jsons_local:
                coco_path = coco_jsons_local[0]
            if coco_path is None:
                raise RuntimeError(
                    "Не удалось автоматически найти маски или COCO JSON. "
                    "Откройте структуру датасета и укажите пути вручную в этом классе."
                )
            coco = json.loads(coco_path.read_text(encoding="utf-8"))
            imgs = {im["id"]: im for im in coco.get("images", [])}
            cats = sorted({c["id"] for c in coco.get("categories", [])})
            self.coco_categories = cats
            self.coco = coco
            self.coco_path = coco_path
            self.mode = "coco"
            id_to_file = {i: Path(coco_path).parent / im["file_name"] for i, im in imgs.items()}
            # если file_name относительный — попробуем несколько баз
            fixed = {}
            for i, p in id_to_file.items():
                if p.is_file():
                    fixed[i] = p
                    continue
                alt = DATA_ROOT / imgs[i]["file_name"]
                if alt.is_file():
                    fixed[i] = alt
                    continue
                # последний fallback: поиск по имени файла
                name = Path(imgs[i]["file_name"]).name
                hits = list(DATA_ROOT.rglob(name))
                fixed[i] = hits[0] if hits else p
            self.items = [(fixed[i], None, coco_path, i) for i in fixed.keys()]

        # простое разбиение train/val по имени файла (детерминированно)
        rng = random.Random(42)
        idxs = list(range(len(self.items)))
        rng.shuffle(idxs)
        cut = int(0.85 * len(idxs))
        chosen = idxs[:cut] if split == "train" else idxs[cut:]
        self.items = [self.items[i] for i in chosen]

        if max_samples is not None:
            self.items = self.items[:max_samples]

        # вычислим число классов (включая фон = 0 в loss мы обработаем отдельно)
        if self.mode == "png":
            max_id = 0
            for img_p, mask_p, _, _ in tqdm(self.items, desc="Scan masks for classes", leave=False):
                m = np.array(Image.open(mask_p).convert("L"))
                max_id = max(max_id, int(m.max()))
            self.num_classes = max(2, max_id + 1)  # если маска 0/1
        else:
            self.num_classes = len(self.coco_categories) + 1  # +фон

        self.img_tf = T.Compose(
            [
                T.Resize((image_size, image_size), interpolation=T.InterpolationMode.BILINEAR),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        )

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, mask_path, coco_json, image_id = self.items[idx]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        if self.mode == "png":
            mask = np.array(Image.open(mask_path).convert("L")).astype(np.int64)
        else:
            coco = json.loads(Path(coco_json).read_text(encoding="utf-8"))
            mask = rasterize_coco_to_semantic(coco, image_id, (h, w), self.coco_categories)

        mask_pil = Image.fromarray(mask.astype(np.uint8), mode="L")
        img = T.Resize((self.image_size, self.image_size), interpolation=T.InterpolationMode.BILINEAR)(img)
        mask_pil = T.Resize((self.image_size, self.image_size), interpolation=T.InterpolationMode.NEAREST)(mask_pil)
        x = self.img_tf(img)
        y = torch.from_numpy(np.array(mask_pil)).long()
        return x, y


# Для отладки на слабых машинах уменьшите max_samples; для отчёта поставьте None
MAX_SAMPLES = 400  # None = весь доступный split

train_ds = StrawberrySemanticDataset(DATA_ROOT, image_size=256, split="train", max_samples=MAX_SAMPLES)
val_ds = StrawberrySemanticDataset(DATA_ROOT, image_size=256, split="val", max_samples=max(50, MAX_SAMPLES // 5 if MAX_SAMPLES else 200))
num_classes = train_ds.num_classes
print("num_classes (включая фон как 0 в маске):", num_classes)
print("train:", len(train_ds), "val:", len(val_ds), "mode:", train_ds.mode)

In [ ]:
def compute_metrics(logits: torch.Tensor, target: torch.Tensor, num_classes: int):
    """logits: NCHW, target: NHW с значениями 0..num_classes-1"""
    pred = logits.argmax(dim=1)
    pred = pred.view(-1)
    tgt = target.view(-1)
    valid = (tgt >= 0) & (tgt < num_classes)
    pred = pred[valid]
    tgt = tgt[valid]

    pixel_acc = (pred == tgt).float().mean().item()

    ious = []
    dices = []
    for c in range(num_classes):
        tp = ((pred == c) & (tgt == c)).sum().float()
        fp = ((pred == c) & (tgt != c)).sum().float()
        fn = ((pred != c) & (tgt == c)).sum().float()
        denom = tp + fp + fn
        iou = (tp / denom.clamp(min=1.0)).item() if denom > 0 else float("nan")
        ious.append(iou)
        dice = (2 * tp / (2 * tp + fp + fn).clamp(min=1.0)).item() if (2 * tp + fp + fn) > 0 else float("nan")
        dices.append(dice)

    miou = np.nanmean(np.array(ious, dtype=np.float32))
    mdice = np.nanmean(np.array(dices, dtype=np.float32))
    return {"pixel_acc": pixel_acc, "mIoU": float(miou), "mDice": float(mdice)}


@torch.no_grad()
def evaluate(model, loader, num_classes: int):
    model.eval()
    meters = {"pixel_acc": 0.0, "mIoU": 0.0, "mDice": 0.0}
    n = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        out = model(x)
        logits = out["out"] if isinstance(out, dict) else out
        m = compute_metrics(logits, y, num_classes)
        bs = x.size(0)
        for k in meters:
            meters[k] += m[k] * bs
        n += bs
    return {k: v / max(n, 1) for k, v in meters.items()}

## 2. Бейзлайн: сверточная и «трансформерная» модели через `segmentation_models.pytorch` (SMP)

Библиотека **[segmentation_models.pytorch](https://github.com/qubvel/segmentation_models.pytorch)** задаёт единый API для декодеров (здесь **DeepLabV3+**) и энкодеров из `timm`.

- **Сверточный бейзлайн:** `DeepLabV3+` с энкодером **`resnet50`** и весами ImageNet (`encoder_weights="imagenet"`).
- **Трансформерный бейзлайн:** тот же декодер **`DeepLabV3+`**, но с энкодером **`mit_b0`** — **Mix Transformer** из семейства SegFormer, то есть явно трансформерная архитектура признаков.

**Замечание:** `torchvision` по-прежнему используется только для **`torchvision.transforms`** (нормализация и ресайз в датасете). Все модели сегментации в п.2–3 строятся через SMP.

In [ ]:
def build_smp_cnn(num_classes: int) -> nn.Module:
    """Сверточный энкодер ResNet50 + декодер DeepLabV3+ (SMP)."""
    return smp.DeepLabV3Plus(
        encoder_name="resnet50",
        encoder_weights="imagenet",
        in_channels=3,
        classes=num_classes,
    )


def build_smp_transformer(num_classes: int) -> nn.Module:
    """Трансформерный энкодер MiT-B0 (SegFormer) + декодер DeepLabV3+ (SMP)."""
    return smp.DeepLabV3Plus(
        encoder_name="mit_b0",
        encoder_weights="imagenet",
        in_channels=3,
        classes=num_classes,
    )


def train_one_epoch(model, loader, optimizer, scaler=None):
    model.train()
    losses = []
    for x, y in tqdm(loader, leave=False):
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=scaler is not None):
            out = model(x)
            logits = out["out"] if isinstance(out, dict) else out
            loss = F.cross_entropy(logits, y, ignore_index=255)
        if scaler is None:
            loss.backward()
            optimizer.step()
        else:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        losses.append(loss.item())
    return float(np.mean(losses))


def train_model(model, name: str, epochs: int = 3, batch_size: int = 8, lr: float = 3e-4, use_amp: bool = True):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=bool(use_amp and device.type == "cuda"))

    hist = []
    for ep in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, opt, scaler=scaler if scaler.is_enabled() else None)
        metrics = evaluate(model, val_loader, num_classes)
        row = {"epoch": ep, "train_loss": tr_loss, **metrics}
        hist.append(row)
        print(f"[{name}] ep{ep}: loss={tr_loss:.4f} | " + " ".join([f"{k}={v:.4f}" for k, v in metrics.items()]))
    return hist


BASE_EPOCHS = 3  # увеличьте для отчёта (например, 15–30)
BASE_BS = 8

deeplab = build_smp_cnn(num_classes)
hist_deeplab_base = train_model(deeplab, "SMP DeepLabV3+ ResNet50 (baseline)", epochs=BASE_EPOCHS, batch_size=BASE_BS)

mitseg = build_smp_transformer(num_classes)
hist_mit_base = train_model(mitseg, "SMP DeepLabV3+ MiT-B0 (baseline)", epochs=BASE_EPOCHS, batch_size=max(4, BASE_BS // 2))

## 3. Улучшение бейзлайна: гипотезы, проверка, сравнение с п.2

### 3a) Гипотезы

1. **Аугментации** (горизонтальное отражение, лёгкий photometric jitter) увеличат устойчивость к вариациям освещения и ракурса в теплице/поле.
2. **Планировщик learning rate** (`CosineAnnealingLR`) улучшит сходимость при малых датасетах.
3. **Чуть больший размер входа** (например 288 вместо 256) даст больше деталей границы поражения (цена — память и время).

### 3b–3e) Реализация улучшенного пайплайна

Ниже создаётся `train_ds_aug` с аугментациями; модели переобучаются с тем же числом эпох для честного сравнения в рамках ноутбука (в отчёте лучше зафиксировать одинаковый бюджет эпох/GPU-времени).

In [ ]:
class StrawberrySemanticDatasetAug(StrawberrySemanticDataset):
    def __init__(self, *args, jitter: bool = True, hflip_p: float = 0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.jitter = jitter
        self.hflip_p = hflip_p

    def __getitem__(self, idx):
        img_path, mask_path, coco_json, image_id = self.items[idx]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        if self.mode == "png":
            mask = np.array(Image.open(mask_path).convert("L")).astype(np.int64)
        else:
            coco = json.loads(Path(coco_json).read_text(encoding="utf-8"))
            mask = rasterize_coco_to_semantic(coco, image_id, (h, w), self.coco_categories)

        mask_pil = Image.fromarray(mask.astype(np.uint8), mode="L")

        if random.random() < self.hflip_p:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            mask_pil = mask_pil.transpose(Image.FLIP_LEFT_RIGHT)

        if self.jitter:
            img = T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03)(img)

        img = T.Resize((self.image_size, self.image_size), interpolation=T.InterpolationMode.BILINEAR)(img)
        mask_pil = T.Resize((self.image_size, self.image_size), interpolation=T.InterpolationMode.NEAREST)(mask_pil)

        x = self.img_tf(img)
        y = torch.from_numpy(np.array(mask_pil)).long()
        return x, y


train_ds_aug = StrawberrySemanticDatasetAug(DATA_ROOT, image_size=288, split="train", max_samples=MAX_SAMPLES)
val_ds_aug = StrawberrySemanticDataset(DATA_ROOT, image_size=288, split="val", max_samples=max(50, MAX_SAMPLES // 5 if MAX_SAMPLES else 200))


def train_model_sched(model, name: str, train_dataset, val_dataset, epochs: int = 3, batch_size: int = 8, lr: float = 3e-4, use_amp: bool = True):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=bool(use_amp and device.type == "cuda"))

    hist = []
    for ep in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, opt, scaler=scaler if scaler.is_enabled() else None)
        sched.step()
        metrics = evaluate(model, val_loader, num_classes)
        hist.append({"epoch": ep, "train_loss": tr_loss, **metrics})
        print(f"[{name}] ep{ep}: loss={tr_loss:.4f} | " + " ".join([f"{k}={v:.4f}" for k, v in metrics.items()]))
    return hist


IMP_EPOCHS = BASE_EPOCHS
IMP_BS = 6

deeplab_imp = build_smp_cnn(num_classes)
hist_deeplab_imp = train_model_sched(deeplab_imp, "SMP DeepLabV3+ ResNet50 (improved)", train_ds_aug, val_ds_aug, epochs=IMP_EPOCHS, batch_size=IMP_BS)

mitseg_imp = build_smp_transformer(num_classes)
hist_mit_imp = train_model_sched(mitseg_imp, "SMP DeepLabV3+ MiT-B0 (improved)", train_ds_aug, val_ds_aug, epochs=IMP_EPOCHS, batch_size=max(2, IMP_BS // 2))

print("\nСравнение (последняя эпоха):")
print("ResNet50 baseline:", hist_deeplab_base[-1])
print("ResNet50 improved:", hist_deeplab_imp[-1])
print("MiT-B0 baseline:", hist_mit_base[-1])
print("MiT-B0 improved:", hist_mit_imp[-1])

### 3g) Выводы по разделу 3

В тексте отчёта зафиксируйте: какие гипотезы подтвердились по **mIoU/mDice**, был ли выигрыш у **ResNet50** vs **MiT-B0** (при одинаковом декодере DeepLabV3+), и какие ограничения эксперимента (малый `MAX_SAMPLES`, мало эпох, возможный дисбаланс фона). При необходимости добавьте графики обучения (loss/mIoU по эпохам) — это усиливает воспроизводимость.

## 4. Самостоятельная имплементация моделей, сравнение с п.2 и п.3

Ниже — две **учебные** архитектуры «с нуля» (без претрейна), чтобы выполнить требование «имплементировать модели»:

- **Mini-UNet:** классический энкодер–декодер с skip-connections.
- **Mini-PSPNet-блок:** пирамидальный pooling + декодер (упрощённый вариант идеи PSPNet).

Далее к **улучшенному бейзлайну из п.3с** добавляются **те же аугментации и cosine schedule**, что и для **SMP**-моделей в п.2–3.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class MiniUNet(nn.Module):
    def __init__(self, in_ch: int, num_classes: int, base: int = 32):
        super().__init__()
        self.d1 = DoubleConv(in_ch, base)
        self.d2 = DoubleConv(base, base * 2)
        self.d3 = DoubleConv(base * 2, base * 4)
        self.down = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.u3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.u2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.u1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        c1 = self.d1(x)
        p1 = self.down(c1)
        c2 = self.d2(p1)
        p2 = self.down(c2)
        c3 = self.d3(p2)
        p3 = self.down(c3)
        b = self.bottleneck(p3)
        x = self.up3(b)
        x = self.u3(torch.cat([x, c3], dim=1))
        x = self.up2(x)
        x = self.u2(torch.cat([x, c2], dim=1))
        x = self.up1(x)
        x = self.u1(torch.cat([x, c1], dim=1))
        return {"out": self.head(x)}


class MiniPSPNet(nn.Module):
    def __init__(self, in_ch: int, num_classes: int, base: int = 32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, base, 3, padding=1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(inplace=True),
            nn.Conv2d(base, base * 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(base * 2),
            nn.ReLU(inplace=True),
        )
        mid = base * 2
        psp_out = max(32, mid // 4)
        pool_sizes = (1, 2, 3, 6)
        self.psp_stems = nn.ModuleList(
            [
                nn.Sequential(
                    nn.AdaptiveAvgPool2d(k),
                    nn.Conv2d(mid, psp_out, 1, bias=False),
                    nn.BatchNorm2d(psp_out),
                    nn.ReLU(inplace=True),
                )
                for k in pool_sizes
            ]
        )
        self.psp_head = nn.Sequential(
            nn.Conv2d(mid + psp_out * len(pool_sizes), psp_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(psp_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(psp_out, num_classes, 1),
        )

    def forward(self, x):
        x = self.stem(x)
        h, w = x.shape[-2:]
        feats = [x]
        for stem in self.psp_stems:
            y = stem(x)
            y = F.interpolate(y, size=(h, w), mode="bilinear", align_corners=False)
            feats.append(y)
        z = torch.cat(feats, dim=1)
        logits = self.psp_head(z)
        return {"out": logits}


custom_unet = MiniUNet(in_ch=3, num_classes=num_classes, base=32)
custom_psp = MiniPSPNet(in_ch=3, num_classes=num_classes, base=32)

hist_unet_custom = train_model(custom_unet, "MiniUNet (custom, baseline-like)", epochs=max(2, BASE_EPOCHS), batch_size=BASE_BS)
hist_psp_custom = train_model(custom_psp, "MiniPSP (custom, baseline-like)", epochs=max(2, BASE_EPOCHS), batch_size=BASE_BS)

custom_unet_imp = MiniUNet(in_ch=3, num_classes=num_classes, base=32)
custom_psp_imp = MiniPSPNet(in_ch=3, num_classes=num_classes, base=32)
hist_unet_imp = train_model_sched(custom_unet_imp, "MiniUNet (custom + aug+sched)", train_ds_aug, val_ds_aug, epochs=IMP_EPOCHS, batch_size=IMP_BS)
hist_psp_imp = train_model_sched(custom_psp_imp, "MiniPSP (custom + aug+sched)", train_ds_aug, val_ds_aug, epochs=IMP_EPOCHS, batch_size=IMP_BS)

print("\nИтоговое сравнение (последняя эпоха, короткий прогон — для отчёта увеличьте эпохи):")
rows = [
    ("SMP ResNet50 baseline", hist_deeplab_base[-1]),
    ("SMP MiT-B0 baseline", hist_mit_base[-1]),
    ("SMP ResNet50 improved", hist_deeplab_imp[-1]),
    ("SMP MiT-B0 improved", hist_mit_imp[-1]),
    ("MiniUNet custom", hist_unet_custom[-1]),
    ("MiniPSP custom", hist_psp_custom[-1]),
    ("MiniUNet custom+imp", hist_unet_imp[-1]),
    ("MiniPSP custom+imp", hist_psp_imp[-1]),
]
for name, r in rows:
    print(name, {k: round(float(v), 4) for k, v in r.items() if k != "epoch"})

### Визуализация (пример для отчёта)

Ниже: одно изображение из валидации, эталонная маска и предсказание **улучшенной SMP-модели** DeepLabV3+ с энкодером ResNet50 (после п.3).

In [ ]:
import matplotlib.pyplot as plt


def denormalize_tensor(x: torch.Tensor) -> np.ndarray:
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    a = x.cpu().numpy() * std + mean
    return np.clip(a.transpose(1, 2, 0), 0, 1)


@torch.no_grad()
def show_val_example(model, ds, idx: int = 0):
    model.eval()
    x, y = ds[idx]
    logits = model(x.unsqueeze(0).to(device))
    logits = logits["out"] if isinstance(logits, dict) else logits
    pred = logits.argmax(1).cpu().numpy()[0]
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(denormalize_tensor(x))
    ax[0].set_title("RGB")
    ax[1].imshow(y.numpy(), cmap="tab10", vmin=0, vmax=max(10, num_classes - 1))
    ax[1].set_title("GT mask")
    ax[2].imshow(pred, cmap="tab10", vmin=0, vmax=max(10, num_classes - 1))
    ax[2].set_title("Prediction")
    for a in ax:
        a.axis("off")
    plt.tight_layout()
    plt.show()


show_val_example(deeplab_imp, val_ds_aug, idx=0)

### 4d–4j) Выводы

1. **Сравнение с п.2:** как правило, предобученные **SMP**-модели (ImageNet-энкодер) при малом бюджете обучения выигрывают у «случайно инициализированных» кастомных сетей — это ожидаемо и хорошо объясняется переносом признаков.
2. **Сравнение с п.3:** если аугментации и schedule дали прирост mIoU у ResNet50/MiT, аналогичный приём должен улучшать и кастомные модели — но масштаб эффекта может быть меньше без предобучения.
3. **Практический смысл:** для агромониторинга важнее стабильность на реальных снимках (доменный сдвиг), поэтому в отчёте логично предложить сбор собственных 100–300 размеченных кадров с места внедрения.

---

**Чек-лист перед сдачей:**
- Убрать/увеличить `MAX_SAMPLES`, увеличить `BASE_EPOCHS`/`IMP_EPOCHS`.
- Вставить 2–3 визуализации: исходное изображение, GT-маска, предсказание (можно добавить ячейку с `matplotlib.imshow`).
- Убедиться, что Kaggle credentials работают в Colab.